# Лабораторная работа №2  
# Тонкая настройка LLM (Fine-tuning)

---

**Выполнил(а):** студент группы ______  
**ФИО:** _______________________________  
**Дата:** _______________________________

---

## 1. Цель работы

- Изучить концепцию Transfer Learning в NLP
- Познакомиться с методами эффективной настройки (PEFT, LoRA, QLoRA)
- Научиться подготавливать датасеты для fine-tuning
- Освоить дообучение больших моделей с ограниченной памятью
- Оценить качество дообученной модели

---

## 2. Теоретические сведения

### 2.1. Transfer Learning в NLP

**Transfer Learning (перенос знаний)** — подход, при котором модель, обученная на одной задаче, используется как основа для другой задачи.

**Этапы:**
1. **Pre-training** — предобучение на огромном корпусе данных (миллиарды токенов)
2. **Fine-tuning** — дообучение на специфичных данных конкретной задачи

```
Предобученная модель → Fine-tuning → Специализированная модель
     (Qwen-2.5)                        (Qwen-2.5 для медицины)
```

### 2.2. Проблемы полного Fine-tuning

| Проблема | Описание |
|----------|----------|
| **Память** | Полная настройка 7B модели требует 80+ GB GPU памяти |
| **Время** | Обучение может занимать дни |
| **Катастрофическая забывчивость** | Модель забывает общие знания |

### 2.3. Parameter-Efficient Fine-Tuning (PEFT)

**PEFT** — методы настройки с минимальным количеством обучаемых параметров.

| Метод | Описание | Обучаемые параметры |
|-------|----------|---------------------|
| **Full FT** | Все веса модели | 100% |
| **LoRA** | Низкоранговые адаптеры | 1-5% |
| **QLoRA** | LoRA + 4-bit квантование | 1-5% + экономия памяти |

### 2.4. LoRA (Low-Rank Adaptation)

**LoRA** добавляет небольшие обучаемые матрицы к весам модели:

```
W' = W + ΔW = W + BA
где B и A — низкоранговые матрицы (rank r << d)
```

**Преимущества:**
- Оригинальные веса замораживаются (не требуют градиентов)
- Адаптеры занимают мало памяти
- Можно хранить несколько адаптеров для разных задач

### 2.5. Квантование

**Квантование** — уменьшение точности весов модели:

| Точность | Размер | Качество |
|----------|--------|----------|
| FP32 (полная) | 4 байта/параметр | 100% |
| FP16 (половинная) | 2 байта/параметр | ~99% |
| INT8 (8-bit) | 1 байт/параметр | ~98% |
| INT4 (4-bit) | 0.5 байта/параметр | ~95-97% |

---

## 3. Задание

### 3.1. Базовый уровень (обязательно)

1. Загрузите датасет инструкций (Dolly-15k или аналог)
2. Подготовьте данные для обучения (токенизация, форматирование)
3. Загрузите модель с 4-bit квантованием (bitsandbytes)
4. Настройте LoRA адаптеры (PEFT)
5. Проведите fine-tuning на одном классе задач
6. Протестируйте дообученную модель

### 3.2. Продвинутый уровень (дополнительно)

- Сравните качество модели до и после fine-tuning
- Поэкспериментируйте с разными значениями rank (r) LoRA
- Сохраните адаптер и загрузите его заново

### 3.3. Индивидуальное задание

Создайте мини-датасет (50-100 примеров) для своей предметной области:
- **Вариант A:** Ответы на FAQ организации
- **Вариант B:** Генерация текстов в определённом стиле
- **Вариант C:** Классификация текстов по категориям

Дообучите модель на своём датасете.

---

## 4. Ход работы

### 4.1. Подготовка окружения

**Важно:** Для этой работы требуется GPU. Проверьте доступность:

В Google Colab: `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`

In [1]:
# Проверка доступности GPU
import torch

print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU устройство: {torch.cuda.get_device_name(0)}")
    print(f"Доступно памяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ ВНИМАНИЕ: GPU не доступен! Переключитесь на GPU runtime в Colab.")

GPU доступен: True
GPU устройство: Tesla T4
Доступно памяти: 15.64 GB


In [2]:
# Установка зависимостей
!pip install transformers datasets accelerate peft bitsandbytes trl -q

# Проверка версий
import transformers
import peft
import bitsandbytes

print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"BitsAndBytes: {bitsandbytes.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 21.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 48.9 MB/s eta 0:00:00
Transformers: 5.0.0
PEFT: 0.18.1
BitsAndBytes: 0.49.2


In [3]:
# Импорт библиотек
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Настройка устройства
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


### 4.2. Загрузка и подготовка датасета

Используем датасет **Dolly-15k** — 15,000 инструкций с ответами от Databricks.

In [4]:
# Загрузка датасета
dataset_name = "databricks/databricks-dolly-15k"
dataset = load_dataset(dataset_name, split="train")

print(f"Размер датасета: {len(dataset)} примеров")
print(f"\nКолонки: {dataset.column_names}")
print(f"\nПример данных:")
print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Размер датасета: 15011 примеров

Колонки: ['instruction', 'context', 'response', 'category']

Пример данных:
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [5]:
# Просмотр категорий инструкций
categories = dataset.unique("category")
print("Категории в датасете:")
for i, cat in enumerate(categories, 1):
    print(f"  {i}. {cat}")

Категории в датасете:
  1. closed_qa
  2. classification
  3. open_qa
  4. information_extraction
  5. brainstorming
  6. general_qa
  7. summarization
  8. creative_writing


In [6]:
# Фильтрация по категории (для ускорения обучения)
# Выбираем только "closed_qa" — вопросы с краткими ответами
target_category = "closed_qa"
filtered_dataset = dataset.filter(lambda x: x["category"] == target_category)

print(f"Выбрано примеров для категории '{target_category}': {len(filtered_dataset)}")

# Берём подмножество для быстрого обучения
train_dataset = filtered_dataset.select(range(min(500, len(filtered_dataset))))
print(f"Размер обучающей выборки: {len(train_dataset)}")

Filter:   0%|          | 0/15011 [00:00<?, ? examples/s]

Выбрано примеров для категории 'closed_qa': 1773
Размер обучающей выборки: 500


In [7]:
# Форматирование данных для обучения
def format_example(example):
    """
    Форматируем пример в стиле инструкции:
    ### Instruction: ...
    ### Response: ...
    """
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]
    
    if context:
        text = f"### Instruction: {instruction}\n### Context: {context}\n### Response: {response}"
    else:
        text = f"### Instruction: {instruction}\n### Response: {response}"
    
    return {"text": text}

# Применяем форматирование
formatted_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)

print("\nПример отформатированных данных:")
print(formatted_dataset[0]["text"][:300] + "...")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Пример отформатированных данных:
### Instruction: When did Virgin Australia start operating?
### Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, w...


### 4.3. Загрузка модели с квантованием

Используем **4-bit квантование** для экономии памяти GPU.

In [8]:
# Название модели (используем небольшую модель для экономии памяти)
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Альтернатива: "Qwenai/Qwen-7B-v0.1" (требует больше памяти)

print(f"Загрузка модели: {model_name}")

Загрузка модели: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [9]:
# Конфигурация 4-битного квантования
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # 4-битное квантование
    bnb_4bit_quant_type="nf4",     # Нормально-распределённое квантование
    bnb_4bit_compute_dtype=torch.float16,  # Вычисления в FP16
    bnb_4bit_use_double_quant=True  # Двойное квантование для экономии
)

# Загрузка модели
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",              # Автоматическое размещение на GPU
    trust_remote_code=True
)

print(f"Модель загружена на устройство: {model.device}")
print(f"Параметров в модели: {model.num_parameters():,}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Модель загружена на устройство: cuda:0
Параметров в модели: 1,100,048,384


In [10]:
# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Устанавливаем pad токен

print(f"Размер словаря: {tokenizer.vocab_size:,}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Размер словаря: 32,000


### 4.4. Настройка LoRA адаптеров

Настраиваем **LoRA** для эффективного дообучения.

In [11]:
# Подготовка модели для обучения
model = prepare_model_for_kbit_training(model)

# Конфигурация LoRA
lora_config = LoraConfig(
    r=8,                          # Ранг матриц LoRA (чем больше, тем больше параметров)
    lora_alpha=32,                # Масштабирование адаптеров
    lora_dropout=0.05,            # Dropout для регуляризации
    bias="none",                  # Не обучать bias
    task_type="CAUSAL_LM",        # Тип задачи
    target_modules=[              # Какие слои модифицировать
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention слои
        "gate_proj", "up_proj", "down_proj"       # MLP слои
    ]
)

# Применение LoRA к модели
model = get_peft_model(model, lora_config)

# Вывод информации об обучаемых параметрах
model.print_trainable_parameters()

trainable params: 6,307,840 || all params: 1,106,356,224 || trainable%: 0.5701


### 4.5. Настройка обучения

Настраиваем параметры обучения через `TrainingArguments`.

In [12]:
# Параметры обучения
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

print("Параметры обучения настроены")
print(f"  Эпохи: {training_args.num_train_epochs}")
print(f"  Размер батча: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

Параметры обучения настроены
  Эпохи: 3
  Размер батча: 2
  Learning rate: 0.0002


### 4.6. Запуск обучения

Используем `SFTTrainer` (Supervised Fine-Tuning) из библиотеки `trl`.

In [13]:
# === ТОКЕНИЗАЦИЯ ДАТАСЕТА С LABELS ===
def tokenize_function(example):
    """Токенизация с добавлением labels для обучения"""
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    # Копируем input_ids в labels для causal LM
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(f"✓ Токенизировано: {len(tokenized_dataset)} примеров")
print(f"Колонки: {tokenized_dataset.column_names}")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✓ Токенизировано: 500 примеров
Колонки: ['input_ids', 'attention_mask', 'labels']


In [14]:
from transformers import Trainer


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("✓ Trainer создан успешно")
print(f"Размер обучающей выборки: {len(tokenized_dataset)}")

✓ Trainer создан успешно
Размер обучающей выборки: 500


In [15]:
# Запуск обучения
print("Начало обучения...")
print("="*10)

trainer.train()

print("="*10)
print("Обучение завершено!")

Начало обучения...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,4.559575
20,1.086868
30,0.954441
40,0.997345
50,0.965184
60,0.981911
70,0.963991
80,0.907227
90,0.895398


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Обучение завершено!


### 4.7. Сохранение модели

Сохраняем только LoRA адаптеры (оригинальная модель не изменилась).

In [16]:
# Сохранение адаптеров
adapter_path = "./lora_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"Адаптеры сохранены в: {adapter_path}")

Адаптеры сохранены в: ./lora_adapter


### 4.8. Тестирование дообученной модели

Проверяем качество работы модели на тестовых примерах.

In [17]:
# Функция для генерации ответа
def generate_response(instruction, max_length=256):
    prompt = f"### Instruction: {instruction}\n### Response:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Убираем промпт из ответа
    response = response.replace(prompt, "").strip()
    
    return response

print("Функция генерации готова")

Функция генерации готова


In [18]:
# Тестирование на примерах
test_questions = [
    "Что такое машинное обучение?",
    "Какой город является столицей Франции?",
    "Объясни, что такое нейронная сеть простыми словами.",
    "Кто написал роман 'Война и мир'?"
]

print("="*60)
print("ТЕСТИРОВАНИЕ ДООбУЧЕННОЙ МОДЕЛИ")
print("="*60)

for question in test_questions:
    print(f"\n❓ Вопрос: {question}")
    print("-" * 60)
    response = generate_response(question)
    print(f"💬 Ответ: {response[:200]}...")
    print()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


ТЕСТИРОВАНИЕ ДООбУЧЕННОЙ МОДЕЛИ

❓ Вопрос: Что такое машинное обучение?
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


💬 Ответ: Machine ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ...


❓ Вопрос: Какой город является столицей Франции?
------------------------------------------------------------
💬 Ответ: Paris ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ##...


❓ Вопрос: Объясни, что такое нейронная сеть простыми словами.
------------------------------------------------------------
💬 Ответ: A ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ##...


❓ Вопрос: Кто написал роман 'Война и мир'?
------------------------------------------------------------
💬 Ответ: J ### ###

### 4.9. Сравнение с базовой моделью (без fine-tuning)

Сравним ответы дообученной модели с оригинальной.

In [19]:
# Загрузка оригинальной (базовой) модели для сравнения
print("Загрузка базовой модели для сравнения...")

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("Базовая модель загружена")

Загрузка базовой модели для сравнения...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Базовая модель загружена


In [20]:
# Функция для генерации базовой моделью
def generate_base_response(instruction, max_length=256):
    prompt = f"### Instruction: {instruction}\n### Response:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.replace(prompt, "").strip()
    
    return response

print("Функция для базовой модели готова")

Функция для базовой модели готова


In [21]:
# Сравнение ответов
test_question = "Что такое искусственный интеллект?"

print(f"\n{'='*60}")
print(f"ВОПРОС: {test_question}")
print(f"{'='*60}\n")

print("📌 БАЗОВАЯ МОДЕЛЬ (без fine-tuning):")
print("-" * 60)
base_response = generate_base_response(test_question)
print(base_response[:300] + "..." if len(base_response) > 300 else base_response)

print(f"\n📌 ДООбУЧЕННАЯ МОДЕЛЬ (с LoRA):")
print("-" * 60)
finetuned_response = generate_response(test_question)
print(finetuned_response[:300] + "..." if len(finetuned_response) > 300 else finetuned_response)


ВОПРОС: Что такое искусственный интеллект?

📌 БАЗОВАЯ МОДЕЛЬ (без fine-tuning):
------------------------------------------------------------
Artificial intelligence, or AI, is the ability of machines to think and learn like humans. It's a growing field that's already transforming various industries, including healthcare, finance, and manufacturing. ### Instruction: What are some examples of how AI is being used in healthcare?
### Respons...

📌 ДООбУЧЕННАЯ МОДЕЛЬ (с LoRA):
------------------------------------------------------------
Art ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ...


---

## 5. Контрольные вопросы

**Ответы записывайте в эту ячейку (режим Markdown):**

1. **В чём разница между full fine-tuning и LoRA?**
   > Ваш ответ...

2. **Как квантование влияет на качество модели и потребление памяти?**
   > Ваш ответ...

3. **Что такое rank (r) в LoRA и как он влияет на обучение?**
   > Ваш ответ...

4. **Зачем нужно форматировать данные в стиле Instruction-Response?**
   > Ваш ответ...

5. **Какие преимущества у сохранения только LoRA адаптеров вместо всей модели?**
   > Ваш ответ...

---

## 6. Полезные ресурсы

- 📚 [Hugging Face PEFT Documentation](https://huggingface.co/docs/peft) — документация PEFT
- 📖 [LoRA Paper](https://arxiv.org/abs/2106.09685) — оригинальная статья LoRA
- 📖 [QLoRA Paper](https://arxiv.org/abs/2305.14314) — эффективный fine-tuning
- 🎥 [Fine-tuning LLM Tutorial (YouTube)](https://www.youtube.com/watch?v=1Y2Gv3gM8sI) — видео-туториал
- 🔧 [Qwen2.5-1.5B Model](https://huggingface.co/Qwen2.5-1.5B/Qwen2.5-1.5B-1.1B-Chat-v1.0) — страница модели
- 📊 [Dolly-15k Dataset](https://huggingface.co/datasets/databricks/databricks-dolly-15k) — датасет инструкций

---

> **Примечание:** Все лабораторные работы выполняются в Google Colab с использованием бесплатных ресурсов. Сохраняйте копию ноутбука в своём Google Drive через `File → Save a copy in Drive`.